# UART 诊断操作面板 · 03

已完成的板级验证为：单字节 `55`、64 字节 `00…3F`、BREAK 停止位错误及其后的 `55 AA` 恢复。本页把现有接收驱动封装成按钮操作面板；新面板仍需在你的板上实际运行验收。

**固定配置：115200 baud · 8 数据位 · 无校验 · 1 停止位 · 3.3V TTL · PL 时钟 100 MHz。** 本页不提供可变波特率选项，也不会修改 RTL、驱动或 bit 文件。

将这个唯一新增文件下载到电脑 `C:\Users\cici\Desktop\FPGA\uart_board_stage1\board`，再上传到板上现有 `uart_diag` 目录，与 **`uart_diag.py`、`uart_diag.bit`、`uart_diag.hwh`** 同目录；已有资源无需复制或重新编译。先结束 Windows 发送并保存/备份 `02_uart_receive.ipynb`。启动本页会重载 UART 硬件。再停止 `02` 及其他操作 UART 的内核/脚本，只保留本面板。多个 `03` 面板内核会被互斥锁阻止；旧 `02` 和自写脚本不使用此锁，需要手动停止。

## 使用顺序

1. 停止 Windows 发送端。运行下面的**定义格**与**启动格**，或直接 **Run All**（本页的 Windows 命令都只是说明文本）。
2. 点击 **开始新记录**：已有事件会先自动导出，成功后才清空 FIFO 和本轮计数。请在发送停止时使用此按钮。
3. 去 Windows 发送短突发；发送结束后返回这里，点击 **读取数据**。每次最多读取 256 条；没有自动轮询，FIFO 剩余只在按钮操作时更新。
4. 点击 **导出 CSV**：保存本轮全部记录和同名 JSON 元数据；点击文件链接下载，也可从 Jupyter 文件列表进入 `panel_exports` 下载。重复导出使用新编号，旧文件保留。

表格只显示最近 **200** 条；本轮最多保留 **10000** 条，达到上限后必须导出/开始新记录才能继续读取。硬件 FIFO 只有 **1024** 条；软件记录上限不代表硬件缓存容量，发送过多仍可能溢出。溢出会在面板、CSV 的 `session_overflow_seen` 列和 JSON 中保留。错误行红色，错误帧没有有效字节，显示 `--`，CSV 中数据为空而不是 `00`。ASCII 会按 HTML 文本转义。

时间字段只记录 Python 主机的操作时间，**不是硬件接收时间戳**。CSV 不提供逐字节接收时刻。硬件访问异常后读取与新记录会禁用，已读取的记录仍可导出；随后停止发送并重新运行启动格。重跑启动格会先保存旧实例记录、停用旧按钮，再重载硬件；保存失败则中止重载。


## 1. 定义面板

此格仅定义类和按钮逻辑，不会加载硬件。


In [3]:
import csv, json, fcntl
import importlib.util
import xml.etree.ElementTree as ET
from pathlib import Path
from datetime import datetime, timezone
from html import escape
import ipywidgets as widgets
from IPython.display import display


def utc_now():
    return datetime.now(timezone.utc).isoformat()


class UARTPanel:
    LIMIT, PAGE, BATCH = 10000, 200, 256
    CONFIG = {"baud": 115200, "data_bits": 8, "parity": "none",
              "stop_bits": 1, "io_voltage": "3.3V", "pl_clock_hz": 100000000}

    def __init__(self, rx, base):
        self.rx, self.base, self.records = rx, Path(base), []
        self.failed, self.active, self.overflow = False, True, False
        self.state, self.snapshot_at, self.snapshot_ok = {}, None, False
        self.started_at, self.links = utc_now(), ""
        self.read_button = widgets.Button(description="读取数据", button_style="primary")
        self.new_button = widgets.Button(description="开始新记录")
        self.export_button = widgets.Button(description="导出 CSV")
        self.summary, self.table, self.message = widgets.HTML(), widgets.HTML(), widgets.HTML()
        self.bindings = [(self.read_button, self.on_read), (self.new_button, self.on_new),
                         (self.export_button, self.on_export)]
        for button, callback in self.bindings:
            button.on_click(callback)
        self.ui = widgets.VBox([widgets.HBox([b for b, _ in self.bindings]),
                                self.summary, self.message, self.table])
        self.snapshot()
        self.render("已加载；请停止发送，点击“开始新记录”，再到 Windows 发送。")

    def snapshot(self):
        self.state = self.rx.status()
        self.overflow = self.overflow or self.rx.overflow_seen or self.state["overflow"]
        self.snapshot_at, self.snapshot_ok = utc_now(), True

    def fail(self, error):
        self.failed, self.snapshot_ok = True, False
        self.overflow = self.overflow or self.rx.overflow_seen
        self.render("硬件访问异常：{}。已停止读取/清空；可导出已有记录，之后重跑启动格。".format(error))

    def render(self, message=""):
        valid = sum(bool(e["data_valid"]) for e in self.records)
        errors = sum(bool(e["framing_error"]) for e in self.records)
        remaining = str(self.state.get("count", "未知"))
        if not self.snapshot_ok:
            remaining += "（最后已知值，当前未知）"
        self.summary.value = (
            "<b>固定 115200 / 8N1 / 3.3V</b><br>本轮事件：{} / {}　有效字节：{}　"
            "停止位错误：{}　FIFO 剩余：{}　观察到溢出：<b>{}</b><br>"
            "<small>仅按钮操作时更新；下表显示最近 {} 条，本轮全部记录可导出。</small>"
        ).format(len(self.records), self.LIMIT, valid, errors, remaining,
                 "是（本轮可能丢失事件）" if self.overflow else "否", self.PAGE)
        rows = []
        for e in self.records[-self.PAGE:]:
            good = bool(e["data_valid"]) and not e["framing_error"]
            char = chr(e["data"]) if good and 32 <= e["data"] <= 126 else ("·" if good else "--")
            values = [str(e["index"]), e["hex"] if good else "--", char,
                      "是" if good else "否", "错误" if e["framing_error"] else "正常"]
            style = ' style="color:#a40000;background:#ffe5e5"' if e["framing_error"] else ""
            rows.append("<tr{}>{}</tr>".format(style, "".join(
                "<td style=\"padding:3px 12px\">{}</td>".format(escape(v)) for v in values)))
        self.table.value = ("<table><thead><tr><th>序号</th><th>HEX</th><th>ASCII</th>"
                            "<th>数据有效</th><th>停止位</th></tr></thead><tbody>" +
                            "".join(rows) + "</tbody></table>")
        if len(self.records) >= self.LIMIT:
            message += " 已达本轮 10000 条上限，停止读取；请导出，再开始新记录。"
        if message:
            self.message.value = "<p>{}</p>{}".format(escape(message), self.links)
        self.read_button.disabled = not self.active or self.failed or len(self.records) >= self.LIMIT
        self.new_button.disabled = not self.active or self.failed
        self.export_button.disabled = not self.active

    def save(self):
        # 导出前只读状态；即使硬件失败，仍允许保存已经确认读取的记录。
        if not self.failed:
            try:
                self.snapshot()
            except Exception as error:
                self.fail(error)
        directory = self.base / "panel_exports"
        directory.mkdir(exist_ok=True)
        number = 1
        while True:
            stem = "uart_{:04d}".format(number)
            csv_path, meta_path = directory / (stem + ".csv"), directory / (stem + ".json")
            number += 1
            if meta_path.exists():
                continue
            try:
                stream = csv_path.open("x", encoding="utf-8-sig", newline="")
                break
            except FileExistsError:
                continue
        with stream:
            writer = csv.writer(stream)
            writer.writerow(["index", "data_hex", "data_valid", "framing_error",
                             "overflow_seen_at_read", "session_overflow_seen"])
            for e in self.records:
                writer.writerow([e["index"], e["hex"] if e["data_valid"] else "",
                                 int(e["data_valid"]), int(e["framing_error"]),
                                 int(e["overflow"]), int(self.overflow)])
        metadata = {"configuration": self.CONFIG, "event_count": len(self.records),
                    "valid_byte_count": sum(bool(e["data_valid"]) for e in self.records),
                    "framing_error_count": sum(bool(e["framing_error"]) for e in self.records),
                    "session_overflow_seen": bool(self.overflow),
                    "fifo_remaining": self.state.get("count") if self.snapshot_ok else None,
                    "fifo_remaining_last_known": self.state.get("count"),
                    "fifo_snapshot_at_utc": self.snapshot_at, "hardware_error": self.failed,
                    "session_started_at_utc": self.started_at, "exported_at_utc": utc_now(),
                    "hardware_timestamps": False,
                    "timestamp_note": "时间仅为Python主机操作时间；没有FPGA接收时间戳。",
                    "csv_file": csv_path.name}
        with meta_path.open("x", encoding="utf-8") as stream:
            json.dump(metadata, stream, ensure_ascii=False, indent=2)
        self.links = " · ".join('<a href="/files/uart_diag/panel_exports/{0}" download>{0}</a>'.format(
            escape(p.name)) for p in (csv_path, meta_path))
        return csv_path, meta_path

    def on_read(self, _):
        if not self.active or self.failed or len(self.records) >= self.LIMIT:
            return
        limit, before = min(self.BATCH, self.LIMIT - len(self.records)), len(self.records)
        try:
            # 逐条提交到本轮记录，避免后续POP异常丢掉同批已确认的事件。
            for _ in range(limit):
                batch = self.rx.read_events(max_events=1)
                self.records.extend(batch)
                if not batch:
                    break
            self.snapshot()
        except Exception as error:
            self.fail(error)
            return
        self.render("本次读取 {} 条。{}".format(len(self.records) - before,
                    "FIFO 仍有事件，可继续点击读取。" if self.state["count"] else "FIFO 已读空。"))

    def on_new(self, _):
        if not self.active or self.failed:
            return
        saved = None
        if self.records or self.overflow:
            try:
                saved = self.save()[0].name
            except Exception as error:
                self.render("自动导出失败：{}。未清空记录或 FIFO；请处理后重试。".format(error))
                return
        if self.failed:
            self.render("硬件状态异常；已有记录已导出，未执行清空。请重跑启动格。")
            return
        try:
            state = self.rx.clear()
        except Exception as error:
            self.fail(error)
            return
        self.records, self.state, self.overflow = [], state, bool(self.rx.overflow_seen)
        self.snapshot_at, self.snapshot_ok, self.started_at = utc_now(), True, utc_now()
        self.render(("上一轮已自动保存为 {}。".format(saved) if saved else "") +
                    "新记录已开始；现在去 Windows 发送，发送结束后点击读取。")

    def on_export(self, _):
        if not self.active:
            return
        try:
            paths = self.save()
            self.render("已保存全部 {} 条：{} 及元数据。{}".format(len(self.records), paths[0].name,
                        "硬件异常，读取已禁用；请重跑启动格。" if self.failed else ""))
        except Exception as error:
            self.render("导出失败：{}。已有记录仍在内存中，请处理后重试。".format(error))

    def detach(self):
        self.active = False
        for button, callback in self.bindings:
            button.on_click(callback, remove=True)
            button.disabled = True
        self.message.value = "<p>此面板已停用；请使用最新启动格下方的面板。</p>"


## 2. 启动面板

确认停止发送并关闭其他 UART 内核。此格检查文件、获得独占面板锁，加载现有 Overlay。重跑此格会重载硬件。


In [4]:
# 工作目录必须是本笔记本及 bit/hwh/驱动所在目录。
base = Path.cwd()
for filename in ("uart_diag.py", "uart_diag.bit", "uart_diag.hwh"):
    if not (base / filename).is_file():
        raise FileNotFoundError("当前目录缺少 {}：{}".format(filename, base))
hwh_nodes = ET.parse(str(base / "uart_diag.hwh")).iter()
if not any(n.get("INSTANCE") == "axi_gpio_diag" for n in hwh_nodes):
    raise RuntimeError("hwh 中没有 axi_gpio_diag；请检查配套 bit/hwh。")

# 同一板上只允许一个03面板内核持有锁；关闭该内核会释放锁。
if globals().get("_uart_panel_lock") is None:
    candidate_lock = open("/tmp/pynq_uart_diag_panel.lock", "a")
    try:
        fcntl.flock(candidate_lock, fcntl.LOCK_EX | fcntl.LOCK_NB)
    except OSError:
        candidate_lock.close()
        raise RuntimeError("另一个03面板内核正在使用硬件，请先关闭其内核。")
    _uart_panel_lock = candidate_lock
old_panel = globals().get("_uart_panel")
if old_panel is not None:
    if old_panel.records or old_panel.overflow:
        saved_paths = old_panel.save()  # 保存失败会中止，保留旧面板与记录。
        print("旧面板记录已保存：", *saved_paths)
    old_panel.detach()

from pynq import Overlay
spec = importlib.util.spec_from_file_location("uart_diag_panel_driver", str(base / "uart_diag.py"))
driver = importlib.util.module_from_spec(spec)
spec.loader.exec_module(driver)
_uart_overlay = Overlay(str(base / "uart_diag.bit"))
if "axi_gpio_diag" not in _uart_overlay.ip_dict:
    raise RuntimeError("Overlay 中没有 axi_gpio_diag，请核对 bit/hwh。")

_uart_panel = UARTPanel(driver.UARTDiagnostic(_uart_overlay.axi_gpio_diag.mmio), base)
display(_uart_panel.ui)


## Windows 发送示例（仅说明，不在板上执行）

在 Windows 项目根目录中打开终端。这里使用已经确认的 CH343 端口 **COM5**，保持你原来验证通过的 TX/GND 接线；若设备重新枚举，应先确认实际端口。每轮都按 **面板新记录 → Windows 发送 → 面板读取** 的顺序。

### 正常数据：64 字节 00…3F

```bat
py pc/send_uart.py --port COM5 --ramp
```

期望本轮 64 条事件、64 个有效字节、0 个停止位错误，数据依次为 `00…3F`，FIFO 最后为 0，未观察到溢出。

### BREAK 后恢复：错误事件 + 55 AA

下面命令在 Windows 使用你已配置的 `pyserial`。BREAK 结束后等待 0.1 秒，再发送两个正常字节：

```bat
py -c "import serial,time; s=serial.Serial('COM5',115200,bytesize=8,parity='N',stopbits=1,timeout=1,write_timeout=2); s.send_break(0.02); time.sleep(0.1); s.write(bytes.fromhex('55 AA')); s.flush(); s.close()"
```

沿用已经验证过的 20 毫秒 BREAK 请求。当前接收器在持续低电平期间等待线路恢复，预期为 **1 条红色停止位错误记录，随后 55、AA 两条正常记录，共 3 条**；FIFO 最后为 0，未观察到溢出。20 毫秒是软件请求时长，0.1 秒是人为等待时间，均不代表实测的恢复时间。

操作面板检查完后，可分别导出正常与 BREAK 两轮 CSV/JSON，并保存本 Notebook 的实际输出。验证结论应以板上读到的结果为准。
